In [1]:
import pandas as pd
import geopandas as gpd
import json
import os
import numpy as np

In [2]:
gcs_path = "https://storage.googleapis.com/wetlands-gap-map/original_data"
analysis_data_folder = 'Analysis_Data_insight_Wetlands_Inventory'

In [3]:
locations = gpd.read_file(f'{gcs_path}/locations_simplified.geojson')
locations_sahel = locations[locations['type'] == 'global']
locations_countries = locations[locations['type'] == 'admin_region']
locations_wpda = locations[locations['type'] == 'wdpa']
locations_basins = locations[locations['type'] == 'hydro_basin']
print(f'Total locations: {len(locations)}')
print(f'Total Sahel locations: {len(locations_sahel)}')
print(f'Total country locations: {len(locations_countries)}')
print(f'Total WDPA locations: {len(locations_wpda)}')
print(f'Total basin locations: {len(locations_basins)}')

Total locations: 2068
Total Sahel locations: 1
Total country locations: 26
Total WDPA locations: 2024
Total basin locations: 17


## Wetland type per country

In [4]:
file_path = f"{gcs_path}/{analysis_data_folder}/Country_%25%20wetland%20type_GET%20simplified.shp"
get_wetlands = gpd.read_file(file_path)
get_wetlands['name_en'] = get_wetlands['name_en'].replace({'The Gambia': 'Gambia'})
get_wetlands = get_wetlands.merge(locations_countries[['name', 'code']], left_on='name_en', right_on='name', how='left')
get_wetlands.drop(columns=['name_x', 'name_y', 'fid', 'osm_id', 'boundary', 'admin_leve',
       'admin_cent', 'admin_ce_1', 'admin_ce_2', 'label_node', 'label_no_1',
       'label_no_2', 'layer', 'path', 'HISTO_0', 'HISTO_11', 'HISTO_12',
       'HISTO_13', 'HISTO_14', 'HISTO_15', 'HISTO_16', 'HISTO_17', 'HISTO_18',
       'HISTO_19', 'HISTO_26', 'HISTO_27', 'HISTO_28', 'HISTO_31', 'HISTO_32',
       'HISTO_33', 'HISTO_34', 'HISTO_35', 'HISTO_36', 'HISTO_41', 'HISTO_42', 'tot', 'geometry'],
        inplace=True)
get_wetlands.rename(columns={'GID_0': 'code'}, inplace=True)
get_wetlands.dropna(subset=['code'], inplace=True)
get_wetlands['location_id'] = get_wetlands['code'].apply(lambda x: 'adminregion_' + x.lower())
get_wetlands.drop(columns=['code', 'name_en'], inplace=True)
get_wetlands.columns = get_wetlands.columns.str.lower().str.replace('%', '').str.strip().str.replace(' ', '_')
get_wetlands.head(3)

,permopen,seasopen,ephopen,saline,lake,seas_lake,eph_lake,perinuveg,seainuveg,ephinuveg,agri,reservoir,mangrove,mudflat,location_id
0,35.08,5.03,0.17,0.00,0.10,0.01,0.02,1.07,20.62,6.03,1.02,0.00,30.83,0.033,adminregion_gmb
2,3.99,35.47,16.94,0.22,0.21,0.81,0.15,0.09,14.44,8.71,12.08,6.84,0.00,0.038,adminregion_mrt
3,21.84,17.21,5.22,0.35,3.30,0.25,0.42,1.39,13.76,6.19,8.61,0.81,20.59,0.061,adminregion_sen


In [5]:
get_wetlands_long = get_wetlands.melt(id_vars=['location_id'], var_name='wetland_type', value_name='percentage')
get_wetlands_long.sort_values(['location_id', 'wetland_type'], inplace=True)
get_wetlands_long.head(3)

,location_id,wetland_type,percentage
260,adminregion_ben,agri,10.78
160,adminregion_ben,eph_lake,0.07
235,adminregion_ben,ephinuveg,38.12


In [6]:
get_wetlands_long['wetland_type'].unique()

array(['agri', 'eph_lake', 'ephinuveg', 'ephopen', 'lake', 'mangrove',
       'mudflat', 'perinuveg', 'permopen', 'reservoir', 'saline',
       'seainuveg', 'seas_lake', 'seasopen'], dtype=object)

## Wetland types by wdpa

In [7]:
file_path = f"{gcs_path}/{analysis_data_folder}/WDPA_%25_wetland%20type_GET.shp"
get_wdpa = gpd.read_file(file_path)
get_wdpa.drop(columns=['fid', 'v_idris', 'ramsarid', 'officialna', 'iso3', 'country_en',
       'area_off', 'OBJECTID', 'WDPAID', 'WDPA_PID', 'PA_DEF', 'NAME',
       'ORIG_NAME', 'DESIG', 'DESIG_ENG', 'DESIG_TYPE', 'IUCN_CAT', 'INT_CRIT',
       'MARINE', 'REP_M_AREA', 'GIS_M_AREA', 'REP_AREA', 'GIS_AREA', 'NO_TAKE',
       'NO_TK_AREA', 'STATUS', 'STATUS_YR', 'GOV_TYPE', 'OWN_TYPE',
       'MANG_AUTH', 'MANG_PLAN', 'VERIF', 'METADATAID', 'SUB_LOC',
       'PARENT_ISO', 'SUPP_INFO', 'CONS_OBJ', 'layer', 'path', 'PARENT_I_1',
       'Area_new', 'HISTO_0', 'HISTO_11', 'HISTO_12', 'HISTO_13', 'HISTO_14',
       'HISTO_15', 'HISTO_16', 'HISTO_17', 'HISTO_18', 'HISTO_19', 'HISTO_26',
       'HISTO_27', 'HISTO_28', 'HISTO_31', 'HISTO_32', 'HISTO_33', 'HISTO_34',
       'HISTO_35', 'HISTO_36', 'HISTO_41', 'HISTO_42', 'HISTO_TOT', 'geometry'],
        inplace=True)
get_wdpa['location_id'] = get_wdpa.index.to_series().apply(lambda x: f"wdpa_{x}")
get_wdpa.columns = get_wdpa.columns.str.lower().str.replace('%', '').str.strip().str.replace(' ', '_')
get_wdpa.head()


,permopen,seasopen,epheopen,saltlake,lake,seaslake,ephelake,perinuveg,seasinveg,ephinuveg,agricultu,reseirv,mangrove,mudflat,location_id
0,2.059,7.048,1.475,0.000,0.479,0.849,3.589,0.554,28.110,47.503,0.767,0.055,7.37,0.141,wdpa_0
1,0.024,0.438,0.040,98.734,0.000,0.000,0.000,0.069,0.499,0.196,0.000,0.000,0.00,0.000,wdpa_1
2,0.509,13.549,0.359,78.670,0.000,0.000,0.083,0.071,4.034,2.691,0.034,0.000,0.00,0.000,wdpa_2
3,0.333,9.630,0.713,0.000,83.336,0.393,0.000,0.033,1.062,1.627,2.873,0.000,0.00,0.000,wdpa_3
4,0.682,0.954,0.132,84.302,0.000,0.000,0.000,0.632,5.816,7.342,0.139,0.000,0.00,0.000,wdpa_4


In [8]:
get_wdpa_long = get_wdpa.melt(id_vars=['location_id'], var_name='wetland_type', value_name='percentage')
get_wdpa_long.sort_values(['location_id', 'wetland_type'], inplace=True)
get_wdpa_long.head(3)

,location_id,wetland_type,percentage
20240,wdpa_0,agricultu,0.767
12144,wdpa_0,ephelake,3.589
4048,wdpa_0,epheopen,1.475


In [9]:
types_dict = {
 'agricultu': 'agri',
 'ephelake': 'eph_lake',
 'epheopen': 'ephopen',
 'ephinuveg': 'ephinuveg',
 'lake': 'lake',
 'mangrove': 'mangrove',
 'mudflat': 'mudflat',
 'perinuveg': 'perinuveg',
 'permopen': 'permopen',
 'reseirv': 'reservoir',
 'saltlake': 'saline',
 'seasinveg': 'seainuveg',
 'seaslake': 'seas_lake',
 'seasopen': 'seasopen'
 }
get_wdpa_long['wetland_type'] = get_wdpa_long['wetland_type'].map(types_dict)



## Wetland types by hydrobasin

In [10]:
file_path = f"{gcs_path}/{analysis_data_folder}/Hydrobasins_%25%20wetland%20type_GET.shp"
get_hydrobasins = gpd.read_file(file_path)
get_hydrobasins.drop(columns=['NEXT_DOWN', 'NEXT_SINK', 'MAIN_BAS', 'DIST_SINK',
       'DIST_MAIN', 'SUB_AREA', 'UP_AREA', 'PFAF_ID', 'ENDO', 'COAST', 'ORDER',
       'SORT', 'HISTO_0', 'HISTO_TOT', 'HISTO_11', 'HISTO_12', 'HISTO_13',
       'HISTO_14', 'HISTO_15', 'HISTO_16', 'HISTO_17', 'HISTO_18', 'HISTO_19',
       'HISTO_26', 'HISTO_27', 'HISTO_28', 'HISTO_31', 'HISTO_32', 'HISTO_33',
       'HISTO_34', 'HISTO_35', 'HISTO_36', 'HISTO_41', 'HISTO_42','geometry'],
        inplace=True)
get_hydrobasins['HYBAS_ID'] = get_hydrobasins['HYBAS_ID'].astype(str)

get_hydrobasins = pd.merge(get_hydrobasins, locations_basins[['id', 'code']], left_on='HYBAS_ID', right_on='code', how='left')
get_hydrobasins.drop(columns=['code', 'HYBAS_ID'], inplace=True)
get_hydrobasins.dropna(subset=['id'], inplace=True)
get_hydrobasins.columns = get_hydrobasins.columns.str.lower().str.replace('%', '').str.strip().str.replace(' ', '_')
get_hydrobasins.rename(columns={'id': 'location_id'}, inplace=True)
get_hydrobasins_long = get_hydrobasins.melt(id_vars=['location_id'], var_name='wetland_type', value_name='percentage')


types_dict = {
    'agricult':'agri',
    'ephelake':'eph_lake',
    'ephinveg':'ephinuveg',
    'epheopen':'ephopen',
    'lake':'lake',
    'mangrove':'mangrove',
    'perminveg':'perinuveg',
    'permopen':'permopen',
    'reseivoir':'reservoir',
    'saltlake':'saline',
    'seasinveg':'seainuveg',
    'seaslake':'seas_lake',
    'seasopen':'seasopen',
    'mudflat':'mudflat'
}
get_hydrobasins_long['wetland_type'] = get_hydrobasins_long['wetland_type'].map(types_dict)
get_hydrobasins_long.sort_values(['location_id', 'wetland_type'], inplace=True)
get_hydrobasins_long.head(3)

,location_id,wetland_type,percentage
174,hydrobasin_congo_basin,agri,0.318
106,hydrobasin_congo_basin,eph_lake,0.149
157,hydrobasin_congo_basin,ephinuveg,59.573


## Wetland types Sahel

In [11]:
file_path = f"{gcs_path}/{analysis_data_folder}/Sahel_%25%20wetland%20type_GET%20simplified.shp"
get_sahel = gpd.read_file(file_path)
get_sahel.drop(columns=['fid', 'Id', 'HISTO_0', 'HISTO_11', 'HISTO_12', 'HISTO_13', 'HISTO_14',
       'HISTO_15', 'HISTO_16', 'HISTO_17', 'HISTO_18', 'HISTO_19', 'HISTO_26',
       'HISTO_27', 'HISTO_28', 'HISTO_31', 'HISTO_32', 'HISTO_33', 'HISTO_34',
       'HISTO_35', 'HISTO_36', 'HISTO_41', 'HISTO_42', 'tot', 'geometry'], inplace=True)
get_sahel['location_id'] = 'global_sahel'
get_sahel.columns = get_sahel.columns.str.lower().str.replace('%', '').str.strip().str.replace(' ', '_')
get_sahel_long = get_sahel.melt(id_vars=['location_id'], var_name='wetland_type', value_name='percentage')
get_sahel_long.sort_values(['location_id', 'wetland_type'], inplace=True)
get_sahel_long['wetland_type'] = get_sahel_long['wetland_type'].str.replace('agri_sum', 'agri')
get_sahel_long.head(3)

,location_id,wetland_type,percentage
10,global_sahel,agri,14.16
6,global_sahel,eph_lake,0.12
9,global_sahel,ephinuveg,27.00


In [12]:
types_dict = {
    'agri':'agri',
    'eph_lake':'eph_lake',
    'ephinuveg':'ephinuveg',
    'ephopen':'ephopen',
    'lake':'lake',
    'mangrove':'mangrove',
    'perinuveg':'perinuveg',
    'perm_open':'permopen',
    'reserv':'reservoir',
    'saline':'saline',
    'seasinveg':'seainuveg',
    'seas_lake':'seas_lake',
    'seasopen':'seasopen'
}
get_sahel_long['wetland_type'] = get_sahel_long['wetland_type'].map(types_dict)

In [13]:
print(len(get_wetlands_long['wetland_type'].unique()))
print(len(get_wdpa_long['wetland_type'].unique()))
print(len(get_hydrobasins_long['wetland_type'].unique()))
print(len(get_sahel_long['wetland_type'].unique()))

14
14
14
13


## Combine all data

In [14]:
get_combined = pd.concat([get_wetlands_long, get_wdpa_long, get_hydrobasins_long, get_sahel_long], ignore_index=True)

In [15]:
print(len(get_combined['wetland_type'].unique()))
get_combined['wetland_type'].unique()

14


array(['agri', 'eph_lake', 'ephinuveg', 'ephopen', 'lake', 'mangrove',
       'mudflat', 'perinuveg', 'permopen', 'reservoir', 'saline',
       'seainuveg', 'seas_lake', 'seasopen'], dtype=object)

In [16]:
#Replace Na with 0
get_combined['percentage'] = get_combined['percentage'].replace(np.nan, 0)

In [17]:
color_dict = {'agri':"#ff9067",
              'eph_lake': "#9499ff",
              'ephinuveg': "#9ec59e",
              'ephopen': "#f6ffff",
              'lake': "#000dff",
              'mangrove': "#663399",
              'mudflat': "#a0522d",
              'perinuveg': "#006400",
              'permopen': "#7acaff",
              'reservoir': "#56b98b",
              'saline': "#cfe875",
              'seainuveg': "#8bc500",
              'seas_lake': "#4747ff",
              'seasopen': "#e2ffff"}

In [18]:
indicator_data_list = []

for location in get_combined['location_id'].unique():
    df_location = get_combined[get_combined['location_id'] == location]
    df_location.reset_index(drop=True, inplace=True)
    data_list = []
    for i in range(len(df_location)):
        data_list.append({
            "id":"wetland_type_" + location + "_" + str(i),
            "label": df_location.loc[i, 'wetland_type'],
            "value": df_location.loc[i, 'percentage'],
            "type":"",
            "group": "",
            "color": color_dict.get(df_location.loc[i, 'wetland_type'], "#000000"),
            "format": "number",
            "unit": "%"
        })
    data_json = json.dumps(data_list, indent=2)
    temp_dict = {"id": "wetland-types-" + location,
                 "location": location,
                 "indicator": "wetland-types-get",
                 "data": json.loads(data_json),
                 "locale": {"en": {"labels":{
                     'agri':'Agriculture',
                     'eph_lake':'Ephemeral Lake',
                     'ephinuveg':'Ephemeral Inundated Vegetation',
                     'ephopen':'Ephemeral Open Water',
                     'lake':'Lake',
                     'mangrove':'Mangrove',
                     'mudflat':'Mudflat',
                     'perinuveg':'Permanently Inundated Vegetation',
                     'permopen':'Permanent Open Water',
                     'reservoir':'Reservoir',
                     'saline':'Saline',
                     'seainuveg':'Seasonally Inundated Vegetation',
                     'seas_lake':'Seasonal Lake',
                     'seasopen':'Seasonal Open Water'
                    }
                    }},
                }
    indicator_data_list.append(temp_dict)


In [19]:
print(json.dumps(indicator_data_list[0], indent=2))

{
  "id": "wetland-types-adminregion_ben",
  "location": "adminregion_ben",
  "indicator": "wetland-types-get",
  "data": [
    {
      "id": "wetland_type_adminregion_ben_0",
      "label": "agri",
      "value": 10.78,
      "type": "",
      "group": "",
      "color": "#ff9067",
      "format": "number",
      "unit": "%"
    },
    {
      "id": "wetland_type_adminregion_ben_1",
      "label": "eph_lake",
      "value": 0.07,
      "type": "",
      "group": "",
      "color": "#9499ff",
      "format": "number",
      "unit": "%"
    },
    {
      "id": "wetland_type_adminregion_ben_2",
      "label": "ephinuveg",
      "value": 38.12,
      "type": "",
      "group": "",
      "color": "#9ec59e",
      "format": "number",
      "unit": "%"
    },
    {
      "id": "wetland_type_adminregion_ben_3",
      "label": "ephopen",
      "value": 3.4,
      "type": "",
      "group": "",
      "color": "#f6ffff",
      "format": "number",
      "unit": "%"
    },
    {
      "id": "wetl

### Save to json

In [20]:
seeding_json = json.load(open('../../app-initial-data/indicator-data.json'))
#remove all entries with id that starts with "wetland-types-" and "wetland_types_"
seeding_json = [entry for entry in seeding_json if not entry['id'].startswith('wetland-types-') and not entry['id'].startswith('wetland_types_')]

# Append the new data, if id exists update it
existing_ids = {entry['id'] for entry in seeding_json}
for new_entry in indicator_data_list:
    if new_entry['id'] in existing_ids:
        seeding_json = [entry if entry['id'] != new_entry['id'] else new_entry for entry in seeding_json]
    else:
        seeding_json.append(new_entry)  
with open('../../app-initial-data/indicator-data.json', 'w') as f:
    json.dump(seeding_json, f, indent=2)